<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Appendix B · Ito Calculus Refresher

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/appx_b_ito_calculus_refresher.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

After working through this notebook, you will be able to:

- explain why the non-zero quadratic variation of Brownian motion forces a second-derivative term into the chain rule;
- state Ito's lemma for a process with drift a and diffusion b, and identify the correction as one half of b squared times the second derivative;
- compute the correction for the square transformation and for the logarithm;
- explain why the log drift of a geometric diffusion is the arithmetic drift minus one half of the squared volatility; and
- implement an Euler scheme for a transformed process with and without the correction and measure the difference.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Connect the appendix to a calculation

The appendix explains why transformed fixed-income objects need a corrected chain rule: bond prices written as functions of the short rate in Chapter 12, forward rates written as functions of discount factors, and option values written as functions of state variables in Chapter 14. The companion figure script `code/figures/appx_b_ito_correction.py` plots how the gap between the arithmetic drift and the log drift grows with volatility. This notebook complements the derivation with an Euler experiment: it simulates one log path twice with the same standard normal draws, once with drift mu and once with the corrected drift mu minus one half of the squared volatility, and shows that the two terminal log values differ by exactly the correction accumulated over the horizon.

In [ ]:
import pandas as pd
figure_scripts = sorted((ROOT / "code" / "figures").glob("appx_b_*.py"))
pd.DataFrame({"figure_script": [p.name for p in figure_scripts]})

## Review prompts

Answer these before using the appendix result in a chapter calculation:

- Is the transformation applied to an Ito process, and are the required derivatives of the function smooth where you evaluate them?
- Which term in the transformed drift is the correction, and what is its sign for your function: positive for the square, negative for the logarithm?
- What are the units of the drift and the diffusion, and does the correction carry the same units as the transformed drift?
- Under which measure is the drift stated: a real-world measure for risk analysis or a pricing measure for valuation?
- In your discretization, does the error from omitting the correction stay small relative to the diffusion noise, or does it dominate the quantity of interest?

## Mini-lab: Euler scheme with and without the Ito correction

This lab isolates one mechanism: the Ito correction shifts the drift of a transformed process. A geometric diffusion with arithmetic drift of 5 percent per year and volatility of 20 percent per year is discretized over 252 steps on a one-year horizon. The same standard normal draws drive both schemes, so the entire difference between the two terminal log values is the accumulated correction of one half of the squared volatility times the horizon, exactly the quantity a naive chain rule drops. The simulated path is a fixed-seed illustration, not an estimate.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
mu, sigma, T, steps = 0.05, 0.20, 1.0, 252
dt = T / steps
z = rng.standard_normal(steps)
diffusion = sigma * np.sqrt(dt) * z  # shared diffusion term
log_naive = np.sum(mu * dt + diffusion)  # without correction
log_ito = np.sum((mu - 0.5 * sigma ** 2) * dt + diffusion)
pd.Series({"log_naive": log_naive, "log_ito": log_ito,
           "correction": 0.5 * sigma ** 2 * T})

### What to observe

With seed 7, the naive log scheme ends at -0.4941 and the corrected scheme at -0.5141, a gap of exactly 0.02, which equals one half of the squared volatility over the one-year horizon for sigma of 0.20. The corresponding levels are 0.610 and 0.598, so omitting the correction overstates the terminal value by about 2 percent.

## Exercises

1. **Reproduce the squared-process example.** Apply Ito's lemma to the square transformation for a process with drift 0.02 and diffusion 0.10 at X equals 3, and match the total drift of 0.13 and diffusion coefficient of 0.60. *Hint:* The correction is the squared diffusion, 0.01 here.
2. **Vary volatility.** Rerun the Euler experiment with sigma of 0.10, 0.20, and 0.40 and confirm that the log gap equals one half of the squared volatility each time. *Hint:* Keep the seed and the number of steps fixed.
3. **Average over paths.** Repeat the experiment over many paths and compare the average log drift with and without the correction. *Hint:* The correction shifts the mean, not the variance, of the log increments.
4. **Transform a rate.** Suppose a short rate follows a drift-diffusion process and you need the dynamics of the squared rate; state which derivative term the ordinary chain rule misses. *Hint:* The second derivative of the square is the constant 2.
5. **Link to bond prices.** Explain how Ito's lemma produces the bond-price dynamics behind the no-arbitrage pricing in Chapter 12. *Hint:* The price inherits both the rate drift and a convexity term through the second derivative.

For more detail, see Appendix B of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">